<a href="https://colab.research.google.com/github/sandra-flemse/EpicSortingBattle/blob/master/Book_sorter_by_themes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 13.0 Book Sorting based on Themes

Here we will create a 'book sorter' based on the themes you have specified. The process will include:

1.  **Defining Fiction Themes**: A list of categories you want to sort books by.
2.  **Preparing Book Data**: A simulation of book titles and short descriptions.
3.  **Embedding Themes and Book Descriptions**: Use `SentenceTransformer` to convert both themes and book descriptions into numerical vectors.
4.  **Calculating Cosine Similarity**: Measure the similarity between each book description and each theme.
5.  **Assigning Themes**: Assign the themes with the highest similarity to each book.

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Defining fiction themes
book_themes = [
    "fantasy", "sci-fi", "romance", "horror", "thriller",
    "børne bøger", "young adult", "krimi", "gys",
    "drama", "Psychological", "mystery", "adventure",
    "historical fiction", "biography", "self-help", "science", "poetry", "young adult fiction"
]

# 2. Preparing book data (example data)
fiction_books = pd.DataFrame({
    'title': [
        'The Hobbit', 'Dune', 'Pride and Prejudice',
        'Dracula', 'Gone Girl', "Harry Potter and the Philosopher's Stone",
        'The Hunger Games', 'The Silent Patient', 'The Girl on the Train',
        'It', 'A Streetcar Named Desire', 'Mindhunter',
        'The Witcher: The Last Wish', 'Neuromancer', 'To Kill a Mockingbird',
        'The Haunting of Hill House', 'The Da Vinci Code', 'Ready Player One',
        'The Lord of the Rings', 'Foundation', 'Eleanor Oliphant Is Completely Fine',
        'Mexican Gothic', 'The Guest List',
        'Sapiens: A Brief History of Humankind', 'Cosmos', 'Where the Sidewalk Ends',
        'Educated', 'The Midnight Library', 'Project Hail Mary', 'Becoming',
        'Circe', 'The Henna Artist', 'Normal People', 'The Vanishing Half'
    ],
    'author': [
        'J.R.R. Tolkien', 'Frank Herbert', 'Jane Austen',
        'Bram Stoker', 'Gillian Flynn', 'J.K. Rowling',
        'Suzanne Collins', 'Alex Michaelides', 'Paula Hawkins',
        'Stephen King', 'Tennessee Williams', 'John E. Douglas',
        'Andrzej Sapkowski', 'William Gibson', 'Harper Lee',
        'Shirley Jackson', 'Dan Brown', 'Ernest Cline',
        'J.R.R. Tolkien', 'Isaac Asimov', 'Gail Honeyman',
        'Silvia Moreno-Garcia', 'Lucy Fokley',
        'Yuval Noah Harari', 'Carl Sagan', 'Shel Silverstein',
        'Tara Westover', 'Matt Haig', 'Andy Weir', 'Michelle Obama',
        'Madeline Miller', ' Alka Joshi', 'Sally Rooney', 'Brit Bennett'
    ],
    'description': [
        'A classic tale of a hobbit\'s adventure to reclaim treasure from a dragon, full of magic and mythical creatures.',
        'A sprawling epic of political intrigue, environmentalism, and a messianic figure on a desert planet.',
        'A timeless romance novel exploring societal norms and love among the English gentry.',
        'A gothic horror masterpiece about a vampire\'s reign of terror in England.',
        'A psychological thriller about a woman who disappears on her fifth wedding anniversary, leaving her husband as the prime suspect.',
        'The first book in a beloved series about a young wizard\'s magical education and battle against evil.',
        'A dystopian young adult novel where teenagers are forced to fight to the death in a televised event.',
        'A gripping psychological thriller about a famous painter who shoots her husband and then never speaks another word.',
        'A mystery thriller told from the perspective of an unreliable narrator, focusing on a woman\'s disappearance.',
        'A terrifying horror novel about a monstrous entity that preys on children\'s fears in a small town.',
        'A powerful drama exploring desire, illusion, and the decay of the American South.',
        'A true-crime psychological thriller detailing the early days of the FBI\'s criminal profiling program.',
        'A collection of short stories introducing Geralt of Rivia, a monster hunter with supernatural abilities in a medieval fantasy world.',
        'The foundational work of the cyberpunk genre, featuring a washed-up hacker hired for one last job.',
        'A classic American novel addressing racial injustice and moral growth through the eyes of a young girl.',
        'A seminal work of modern horror literature, exploring the psychological terrors within a haunted house.',
        'A suspenseful mystery thriller involving symbology, art history, and a secret society.',
        'A science fiction adventure set in a dystopian future where humanity escapes reality in a vast virtual world.',
        'An epic high-fantasy adventure following Frodo Baggins on his quest to destroy the One Ring.',
        'A foundational science fiction series chronicling the fall and rebirth of a galactic empire.',
        'A heartwarming yet poignant novel about a socially awkward woman\'s journey to self-discovery and connection.',
        'A gothic horror novel set in 1950s Mexico, blending suspense with dark fantasy.',
        'A classic whodunit mystery set at a remote island wedding, where secrets and danger abound.',
        'A non-fiction book that explores the history of humankind, from the Stone Age to the 21st century.',
        'A classic non-fiction book that explores the universe and our place within it, blending science and philosophy.',
        'A collection of poems and drawings for children and adults, known for its whimsical and imaginative style.',
        'A memoir about a young woman\'s journey from a fundamentalist upbringing in the mountains of Idaho to earning a PhD from Cambridge University.',
        'A novel about a woman who gets a chance to revisit different versions of her life, exploring themes of regret and possibility.',
        'A science fiction novel about an astronaut who is the last hope for humanity, tasked with saving Earth from an alien threat.',
        'A memoir by the former First Lady of the United States, chronicling her life from childhood to her time in the White House.',
        'A mythological novel retelling the story of Circe, a powerful sorceress from Greek mythology, known for transforming men into animals.',
        'A historical fiction novel set in 1950s Jaipur, following a young woman who becomes a famous henna artist and confidante to the city\'s elite.',
        'A coming-of-age romance novel about the complicated relationship between two young people in Ireland.',
        'A novel exploring the lives of twin sisters who choose to live in vastly different worlds, one passing as white, the other remaining in their Black community.'
    ]
})

print("Book data and themes are defined.")

Book data and themes are defined.


In [4]:
pip install -q gradio

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure the model is loaded
# If 'model' from previous cells is still available, reuse it.
# Otherwise, load it again.
if 'model' not in locals():
    model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Embedding themes and book descriptions
theme_vectors = model.encode(book_themes, show_progress_bar=True)
book_description_vectors = model.encode(fiction_books['description'].tolist(), show_progress_bar=True)

print("Themes and book descriptions are now embedded.")

# 4. Calculating Cosine Similarity
# Calculates similarity between each book description and each theme
similarity_matrix = cosine_similarity(book_description_vectors, theme_vectors)

# 5. Assigning themes to books
def assign_top_themes(row_similarity_scores, themes, num_top_themes=2):
    # Get indices of the top N most similar themes
    top_indices = row_similarity_scores.argsort()[-num_top_themes:][::-1]
    # Return the names of these themes and their similarity scores
    return [
        (themes[i], row_similarity_scores[i])
        for i in top_indices
    ]

fiction_books['assigned_themes'] = [
    assign_top_themes(scores, book_themes)
    for scores in similarity_matrix
]

print("Top themes have been assigned to each book.")

# Display the results
print("\nBook Sorting Results:")
for index, row in fiction_books.iterrows():
    themes_str = ', '.join([f"{theme} ({score:.2f})" for theme, score in row['assigned_themes']])
    print(f"Title: {row['title']}\n  Author: {row['author']}\n  Themes: {themes_str}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Themes and book descriptions are now embedded.
Top themes have been assigned to each book.

Book Sorting Results:
Title: The Hobbit
  Author: J.R.R. Tolkien
  Themes: adventure (0.49), historical fiction (0.37)

Title: Dune
  Author: Frank Herbert
  Themes: historical fiction (0.42), young adult fiction (0.33)

Title: Pride and Prejudice
  Author: Jane Austen
  Themes: romance (0.46), historical fiction (0.44)

Title: Dracula
  Author: Bram Stoker
  Themes: thriller (0.52), horror (0.44)

Title: Gone Girl
  Author: Gillian Flynn
  Themes: thriller (0.42), romance (0.29)

Title: Harry Potter and the Philosopher's Stone
  Author: J.K. Rowling
  Themes: young adult fiction (0.44), historical fiction (0.32)

Title: The Hunger Games
  Author: Suzanne Collins
  Themes: young adult fiction (0.56), young adult (0.35)

Title: The Silent Patient
  Author: Alex Michaelides
  Themes: thriller (0.50), young adult fiction (0.40)

Title: The Girl on the Train
  Author: Paula Hawkins
  Themes: thrille

In [7]:
import gradio as gr

def search_books_by_theme(selected_themes):
    if not selected_themes:
        return "Please select at least one theme."

    # Filter books based on whether any of their assigned themes match the selected themes
    filtered_books = fiction_books[
        fiction_books['assigned_themes'].apply(
            lambda x: any(theme_name in selected_themes for theme_name, _ in x)
        )
    ]

    if filtered_books.empty:
        return "No books found for the selected themes."

    output_str = ""
    for index, row in filtered_books.iterrows():
        themes_str = ', '.join([f"{theme} ({score:.2f})" for theme, score in row['assigned_themes']])
        output_str += f"Title: {row['title']}\n"
        output_str += f"Author: {row['author']}\n"
        output_str += f"Description: {row['description']}\n"
        output_str += f"Assigned Themes: {themes_str}\n\n"

    return output_str

In [8]:
iface = gr.Interface(
    fn=search_books_by_theme,
    inputs=gr.CheckboxGroup(choices=book_themes, label="Select Themes"),
    outputs=gr.Textbox(label="Matching Books"),
    title="Book Sorter by Themes",
    description="Select one or more themes to see matching books."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2308d24d08723cf27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
